# TabDPT Regressor — DIMER artifact inference tutorial

[GitHub](https://github.com/kurtvalcorza/tabdpt-regressor-pipeline) · [Open in Colab](https://colab.research.google.com/github/kurtvalcorza/tabdpt-regressor-pipeline/blob/main/tutorials/tabdpt_regressor_artifact_inference_colab.ipynb) · [Model](https://huggingface.co/Layer6/TabDPT)

**Profile:** `ARTIFACT-INFERENCE` · **DIMER Notebook Specification:** 1.0

This notebook consumes `artifact.json` + `training_context.parquet` produced **outside this execution**, validates the artifact and exact pinned base-model provenance, restores fitted preprocessing/support state, accepts genuinely new unlabelled data, predicts continuous values, and exports results. No gradient fine-tuning occurs; reconstruction may call upstream `fit()` to register in-context support, while preprocessing is restored rather than refit.

**Trust boundary.** Digest/manifest checks establish internal consistency, not sender authenticity. This notebook accepts no ZIP, pickle, or arbitrary Python-object artifact. The context is Parquet and the exact Safetensors base weight is acquired separately and digest-verified. Use only artifacts from a trusted producer.

**Prerequisites:** external artifact pair; separate CSV/Parquet new input; Python 3.10+; GPU recommended/CPU slower; FlashAttention disabled for T4 portability. Uploaded data remains in the runtime; do not upload restricted data to an unauthorized environment. This tutorial does not create its own artifact, claim quality without labelled evaluation data, expose calibrated uncertainty, or support classification.


In [ ]:
from pathlib import Path
import shutil
REPO_DIR=Path("/content/tabdpt-regressor-pipeline")
if REPO_DIR.exists(): shutil.rmtree(REPO_DIR)
!git clone -q https://github.com/kurtvalcorza/tabdpt-regressor-pipeline.git /content/tabdpt-regressor-pipeline
%pip install -q -r /content/tabdpt-regressor-pipeline/tutorials/requirements-colab.txt
%pip install -q --no-deps /content/tabdpt-regressor-pipeline
!git -C /content/tabdpt-regressor-pipeline rev-parse HEAD


## 1. Runtime, external artifact upload, and pre-load validation

Upload exactly `artifact.json` and `training_context.parquet`. Validation occurs before model-state reconstruction and checks format/task, support-table path/size/SHA-256, serialized preprocessing consistency, and the manifest's base-model repo/revision/filename/digest/upstream commit against this repository contract. Because external artifact consumption is the purpose of this profile, the upload is the primary path.


In [ ]:
import csv,io,json,platform
import importlib.metadata as md
import pandas as pd, torch
from google.colab import files
from tabdpt_regressor_pipeline import TabDPTRegressionPipeline,load_verified_artifact,validate_artifact_bundle
print("Python",platform.python_version(),"torch",md.version("torch"),"tabdpt",md.version("tabdpt"),
      "Device",torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU","use_flash=False")
ART=Path("/content/external-tabdpt-artifact")
if ART.exists(): shutil.rmtree(ART)
ART.mkdir(parents=True)
up=files.upload(); required={"artifact.json","training_context.parquet"}
if set(up)!=required: raise ValueError(f"Upload exactly {sorted(required)}; got {sorted(up)}")
for name,payload in up.items(): (ART/name).write_bytes(payload)
manifest_path=ART/"artifact.json"
manifest,context_path=validate_artifact_bundle(manifest_path)
print("Validated",manifest["format"],manifest["taskType"]); print(json.dumps(manifest["baseModel"],indent=2))
print("Expected features:",manifest["preprocessing"]["encoder"]["featureColumns"])


## 2. Reconstruct serving state without preprocessing refit

The artifact stores support/preprocessing state rather than a duplicate pretrained checkpoint. After provenance validation, `load_verified_artifact()` acquires the exact pinned base model (or can use an explicitly supplied trusted local copy), verifies the weight digest, restores the serialized encoder/category maps, and conditions TabDPT on the saved support table. This is in-context serving reconstruction, not gradient training.


In [ ]:
pipe=load_verified_artifact(manifest_path,compile_model=False,use_flash=False)
print("Target:",pipe.target_column,"features:",len(pipe.feature_encoder.feature_columns))


## 3. Upload, validate, and score genuinely new input

Upload one CSV or Parquet containing exactly the feature columns printed above, with no target or pre-existing `prediction`. CSV headers are checked before pandas parsing to reject duplicates. The repository then enforces the fitted schema; unseen categories follow the serialized unknown-category policy. No preprocessing is refit from inference data.


In [ ]:
newup=files.upload()
if len(newup)!=1: raise ValueError("Upload exactly one CSV or Parquet input.")
input_name,raw=next(iter(newup.items()))
if input_name.lower().endswith(".csv"):
 text=raw.decode("utf-8-sig"); header=next(csv.reader(io.StringIO(text)),[])
 dup=sorted({x for x in header if header.count(x)>1})
 if dup: raise ValueError(f"Duplicate CSV columns: {dup}")
 new_data=pd.read_csv(io.BytesIO(raw))
elif input_name.lower().endswith((".parquet",".pq")):
 new_data=pd.read_parquet(io.BytesIO(raw),engine="pyarrow")
else: raise ValueError("Input must be CSV or Parquet.")
if new_data.columns.duplicated().any(): raise ValueError("Duplicate columns are not supported.")
if pipe.target_column in new_data or "prediction" in new_data: raise ValueError("Remove target/prediction columns before inference.")
expected=list(pipe.feature_encoder.feature_columns)
missing=[x for x in expected if x not in new_data]; extra=[x for x in new_data if x not in expected]
if missing or extra: raise ValueError(f"Feature schema mismatch; missing={missing}, extra={extra}")
kw={"n_ensembles":2,"context_size":512,"batch_size":512,"seed":42}
pred=pipe.predict(new_data,**kw)
results=pd.DataFrame({"row_id":new_data.index.to_numpy(),"prediction":pred.to_numpy()})
results.head()


## 4. Export predictions and provenance

Predictions are point estimates, not calibrated uncertainty intervals. `row_id` maps each prediction to its input. Provenance records the externally supplied artifact identity, model contract, runtime, and inference configuration; it contains no credentials.


In [ ]:
OUT=Path("/content/tabdpt-artifact-inference-output"); OUT.mkdir(parents=True,exist_ok=True)
results.to_csv(OUT/"tabdpt_regression_predictions.csv",index=False)
prov={"artifact":{"format":manifest["format"],"baseModel":manifest["baseModel"],
"trainingContextSha256":manifest["trainingContext"]["sha256"]},
"runtime":{"python":platform.python_version(),"torch":md.version("torch"),"tabdpt":md.version("tabdpt"),
"device":torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU","use_flash":False},
"inference":kw,"input":{"filename":input_name,"rows":len(new_data),"features":list(new_data.columns)}}
(OUT/"tabdpt_regression_inference_provenance.json").write_text(json.dumps(prov,indent=2)+"\n")
print("Wrote predictions CSV and provenance JSON.")


## Interpretation and troubleshooting

A successful run proves an independently supplied artifact is internally consistent with this repository's pinned model contract, reconstructs saved support/preprocessing state, and scores schema-compatible new records without preprocessing refit. It does **not** authenticate the producer or establish predictive quality, robustness, calibration, fairness, or production fitness. Never bypass a failed manifest/digest/schema check; obtain a correct trusted artifact. If base-model acquisition fails, use only the exact trusted `tabdpt1_2.safetensors` matching the repository/manifest digest rather than substituting another model.
